# **Mục tiêu**

Notebook này được viết nhằm thử nghiệm thiết kế một pipeline cho bộ dữ liệu MIMIC-IV giúp quản lý các bảng, làm sạch dữ liệu giám sát code cũng như chuẩn bi nguyên liệu cho các bước huấn luyện bài toán khác nhau

# **Kiến Trúc Công Nghệ**

Để tối ưu hóa hiệu năng xử lý trên môi trường cá nhân và Google Colab mà không cần đến cụm máy chủ cồng kềnh, hệ thống pipeline này lựa chọn một bộ công cụ tối giản nhưng cực kỳ mạnh mẽ:

| Thành phần | Công nghệ lựa chọn | Giải pháp thay thế (Production) |
|---|---|---|
| **Programming Language** | **Python 3.10+** | Python / Scala |
| **Data Processing Engine** | **DuckDB** | Apache Spark / Databricks |
| **Storage Format** | **Parquet (Apache)** | Delta Lake / Iceberg |
| **Configuration Layer** | **PyYAML (YAML Config)** | Hydra / Decouple |

---

# **Lý Do Lựa Chọn, Ưu Điểm & Nhược Điểm**

## **1. Data Processing Engine: DuckDB**
> **DuckDB** được mệnh danh là "SQLite dành cho phân tích dữ liệu". Nó là một hệ quản trị cơ sở dữ liệu dạng cột (columnar) nhúng trực tiếp vào tiến trình Python, cho phép truy vấn SQL tốc độ cao trên các file dữ liệu lớn mà không cần cài đặt server độc lập.

*   **Tại sao lại sử dụng?** Bảng dữ liệu lâm sàng của MIMIC-IV (như `chartevents`) lên tới hơn 300 triệu dòng (nặng khoảng hơn 30GB thô). Nếu dùng Pandas, hệ thống sẽ cố nạp toàn bộ vào RAM dẫn đến sập nguồn do lỗi `Out of Memory`. DuckDB có khả năng xử lý stream qua ổ đĩa (Out-of-Core processing), giúp xử lý file 30GB mượt mà trên máy chỉ có 8GB RAM.
*   **Ưu điểm:**
    *   **Siêu nhẹ & Tiện lợi:** Cài đặt chỉ với `pip install duckdb`. Không cần cấu hình Java, Hadoop hay Docker cồng kềnh như Apache Spark.
    *   **Tốc độ vượt trội:** Thực thi các câu lệnh SQL (Filter, Aggregation) nhanh gấp hàng chục lần so với Pandas thuần nhờ kiến trúc Vectorized Execution.
    *   **Tích hợp sâu:** Đọc trực tiếp từ file CSV, Parquet và chuyển đổi qua lại với Pandas DataFrame hoặc PyArrow chỉ trong 1 dòng code.
*   **Nhược điểm:**
    *   **Hạn chế mở rộng đa máy (Horizontal Scalability):** DuckDB tối ưu hóa phần cứng trên một máy đơn lẻ (Single-node). Nếu dữ liệu tăng lên quy mô hàng trăm Terabyte/Petabyte trên cụm máy chủ Cloud, nó không thể thay thế cho Apache Spark.

## **2. Storage Format: Apache Parquet**
> **Parquet** là định dạng lưu trữ dữ liệu mã nguồn mở dạng cột (columnar storage), được thiết kế để tối ưu hóa việc lưu trữ và truy vấn trong hệ sinh thái Big Data.

*   **Tại sao lại sử dụng?** Dữ liệu MIMIC-IV được phân phối mặc định ở dạng file `.csv` thô. File CSV lưu dữ liệu theo từng dòng (row-oriented), khiến hệ thống phải quét qua toàn bộ các cột không liên quan khi truy vấn, gây lãng phí tài nguyên đĩa.
*   **Ưu điểm:**
    *   **Tỷ lệ nén cực cao:** Tách biệt dữ liệu theo cột giúp áp dụng các thuật toán nén (như Snappy) hiệu quả hơn. Dung lượng đĩa giảm từ 70% đến 80% so với CSV thô.
    *   **Kỹ thuật Predicate Pushdown:** Cho phép DuckDB đọc trực tiếp siêu dữ liệu (metadata) của file Parquet để bỏ qua các khối dữ liệu (Data Blocks) hoặc các phân vùng không thỏa mãn điều kiện `WHERE` trước khi nạp vào RAM.
*   **Nhược điểm:**
    *   **Không thể đọc bằng mắt thường:** Khác với CSV mở lên là thấy chữ, Parquet là file nhị phân (Binary), muốn xem nội dung bắt buộc phải dùng code hoặc công cụ chuyên dụng (như Parquet Viewer).
    *   **Chi phí ghi đè cao:** Việc cập nhật một vài dòng nhỏ lẻ (Insert/Update) trong file Parquet tốn chi phí hơn vì hệ thống phải ghi lại cả một block cột.

## **3. Configuration Layer: YAML (PyYAML)**
> Sử dụng file cấu hình tách biệt để quản lý toàn bộ tham số của hệ thống (đường dẫn file, danh sách chỉ số xét nghiệm `itemid` cần lọc).

*   **Tại sao lại sử dụng?** Tránh việc "Hard-code" (viết chết đường dẫn hoặc tham số trong mã nguồn). Khi chuyển từ môi trường Google Colab về máy Local, Tiến chỉ cần chỉnh sửa đúng 1 file `.yaml` duy nhất mà không cần chạm vào logic code xử lý của pipeline.
*   **Ưu điểm:** Dễ đọc, cấu trúc phân cấp rõ ràng, dễ bảo trì và mở rộng khi hệ thống phức tạp hơn.
*   **Nhược điểm:** Cần viết thêm code boilerplate nhỏ để parse (đọc) file cấu hình trong Python.

# **Tầng 1: Bronze - Thực hiện việc phân vùng và lưu trữ các file thông qua DuckDB**

- Ý tưởng là thực hiện phân vùng chia nhỏ đối với các file có dung lượng lớn như file `chartevents`, `labelevents` chia theo `itemid` rồi sau đó chuyển đổi các file còn lại thành file parquet giúp cho việc load dễ dàng hơn

In [1]:
# Cài đặt thư viện $ Kết nối Drive
# Cập nhật DuckDB bản mới nhất
!pip install --upgrade duckdb -q

import duckdb
import os
import glob
import time
from google.colab import drive

# Mount Google Drive để đọc/ghi file
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 42.8 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
# Khởi tạo cấu trúc thư mục

# Định nghĩa các đường dẫn giả lập cấu trúc dự án
SOURCE_DIR = "/content/drive/MyDrive/NCKH-DDU1231/mimic-iv-clinical-database-demo-2.2/mimic-iv-clinical-database-demo-2.2" # Đường dẫn file sample trên Drive
BRONZE_BASE_DIR = "./data/bronze"

# Tạo thư mục đầu ra trong môi trường Colab
os.makedirs(BRONZE_BASE_DIR, exist_ok=True)

In [3]:
# Định nghĩa các bảng lớn cần được Partition để tránh quét toàn bộ ổ đĩa
# Các bảng khác nhỏ hơn sẽ được ghi thành 1 file duy nhất cho gọn
PARTITION_STRATEGY = {
    "chartevents": "itemid",
    "labevents": "itemid",
    "prescriptions": "drug"
}

In [4]:
def dynamic_bronze_pipeline(source_dir, output_base_dir):
    # Khởi tạo DuckDB In-Memory Engine
    conn = duckdb.connect(database=':memory:')

    # Tìm tất cả các file .csv.gz nằm trong các thư mục con (recursive=True)
    search_pattern = os.path.join(source_dir, "**/*.csv.gz")
    gz_files = glob.glob(search_pattern, recursive=True)

    if not gz_files:
        print("Không tìm thấy file .csv.gz nào trong thư mục nguồn!")
        return

    print(f"Tìm thấy {len(gz_files)} file nén .csv.gz cần xử lý.\n" + "="*50)

    for file_path in gz_files:
        start_time = time.time()

        # Lấy tên file (ví dụ: admissions.csv.gz) và tên thư mục cha trực tiếp (ví dụ: hosp)
        file_name = os.path.basename(file_path)
        table_name = file_name.replace(".csv.gz", "")
        module_name = os.path.basename(os.path.dirname(file_path))

        # Tạo đường dẫn đầu ra động theo cấu trúc: ./data/bronze/{module_name}/{table_name}
        target_dir = os.path.join(output_base_dir, module_name, table_name)
        os.makedirs(target_dir, exist_ok=True)

        print(f"Đang xử lý: [Module: {module_name}] -> [Bảng: {table_name}]")

        # Kiểm tra xem bảng này có cần áp dụng chiến lược Partition không
        if table_name in PARTITION_STRATEGY:
            partition_column = PARTITION_STRATEGY[table_name]

            # Với bảng demo/sample, ta lấy tất cả dữ liệu có trong file nén
            query = f"""
                COPY (SELECT * FROM read_csv_auto('{file_path}'))
                TO '{target_dir}'
                (FORMAT PARQUET, PARTITION_BY {partition_column}, OVERWRITE_OR_IGNORE 1);
            """
            print(f"Áp dụng Partitioning theo cột: {partition_column}")
        else:
            # Với bảng nhỏ, ghi trực tiếp thành 1 file parquet duy nhất nằm trong thư mục đó
            output_file = os.path.join(target_dir, f"{table_name}.parquet")
            query = f"""
                COPY (SELECT * FROM read_csv_auto('{file_path}'))
                TO '{output_file}'
                (FORMAT PARQUET, OVERWRITE_OR_IGNORE 1);
            """
            print(f"Ghi thành file đơn: {table_name}.parquet")

        # Thực thi câu lệnh qua DuckDB
        try:
            conn.execute(query)
            duration = time.time() - start_time
            print(f"Hoàn thành trong {duration:.2f} giây -> Lưu tại: {target_dir}\n" + "-"*40)
        except Exception as e:
            print(f"Lỗi khi xử lý file {file_name}: {str(e)}\n" + "-"*40)

In [5]:
dynamic_bronze_pipeline(SOURCE_DIR, BRONZE_BASE_DIR)

Tìm thấy 31 file nén .csv.gz cần xử lý.
Đang xử lý: [Module: icu] -> [Bảng: caregiver]
Ghi thành file đơn: caregiver.parquet
Hoàn thành trong 0.55 giây -> Lưu tại: ./data/bronze/icu/caregiver
----------------------------------------
Đang xử lý: [Module: icu] -> [Bảng: ingredientevents]
Ghi thành file đơn: ingredientevents.parquet
Hoàn thành trong 0.98 giây -> Lưu tại: ./data/bronze/icu/ingredientevents
----------------------------------------
Đang xử lý: [Module: icu] -> [Bảng: datetimeevents]
Ghi thành file đơn: datetimeevents.parquet
Hoàn thành trong 0.91 giây -> Lưu tại: ./data/bronze/icu/datetimeevents
----------------------------------------
Đang xử lý: [Module: icu] -> [Bảng: chartevents]
Áp dụng Partitioning theo cột: itemid


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Hoàn thành trong 4.14 giây -> Lưu tại: ./data/bronze/icu/chartevents
----------------------------------------
Đang xử lý: [Module: icu] -> [Bảng: icustays]
Ghi thành file đơn: icustays.parquet
Hoàn thành trong 0.45 giây -> Lưu tại: ./data/bronze/icu/icustays
----------------------------------------
Đang xử lý: [Module: icu] -> [Bảng: d_items]
Ghi thành file đơn: d_items.parquet
Hoàn thành trong 0.56 giây -> Lưu tại: ./data/bronze/icu/d_items
----------------------------------------
Đang xử lý: [Module: icu] -> [Bảng: inputevents]
Ghi thành file đơn: inputevents.parquet
Hoàn thành trong 0.71 giây -> Lưu tại: ./data/bronze/icu/inputevents
----------------------------------------
Đang xử lý: [Module: icu] -> [Bảng: procedureevents]
Ghi thành file đơn: procedureevents.parquet
Hoàn thành trong 0.85 giây -> Lưu tại: ./data/bronze/icu/procedureevents
----------------------------------------
Đang xử lý: [Module: icu] -> [Bảng: outputevents]
Ghi thành file đơn: outputevents.parquet
Hoàn thành t

# **Tầng 2: Silver - Thực hiện việc làm sạch dữ liệu và Chuẩn hóa quan hệ**

Tàng này sẽ kiểm tra chất lượng của dữ liệu biến dữ liệu thô thành các dữ liệu có cấu trúc và đáng tin cậy.

Do đó cần giải quyết 4 bài toán sau:

1. **Ép kiểu dữ liệu (Data Type Casting):** File Parquet ở tầng Bronze được DuckDB tự động nhận diện kiểu dữ liệu `(read_csv_auto)`, nhưng đôi khi chưa tối ưu. Cần ép kiểu tường minh: các cột ID về kiểu INTEGER, các cột thời gian về kiểu TIMESTAMP, các chỉ số về kiểu FLOAT.

2. **Đồng bộ hóa thời gian (Datetime Standardization):** Dữ liệu y tế của MIMIC-IV tràn ngập các cột thời gian (admittime, dischtime, intime, outtime, charttime). Cần đồng bộ tất cả về một chuẩn chung và xử lý các lỗi logic (ví dụ: bộ lọc loại bỏ các dòng có ngày xuất viện trước ngày nhập viện).

3. **Xử lý giá trị khuyết (NULL handling):** Quyết định xem cột nào bắt buộc không được NULL (như các cột khóa chính `subject_id`, `hadm_id`), và cột nào nếu NULL thì xử lý mặc định (ví dụ: nếu kết quả xét nghiệm value là text bị khuyết thì thay bằng 'UNKNOWN').

4. **Tạo các liên kết thực thể (Referential Integrity):** Đảm bảo các bảng dữ liệu lâm sàng (`chartevents, labevents`) đều có thể kết nối (JOIN) mượt mà với bảng xương sống `admissions` và `patients`.

In [6]:
# Định nghĩa các đường dẫn tầng dữ liệu
BRONZE_BASE_DIR = "./data/bronze"
SILVER_BASE_DIR = "./data/silver"

def build_silver_layer(bronze_dir, silver_dir):
    print("[Silver Layer] Bắt đầu quá trình làm sạch và chuẩn hóa dữ liệu...")
    start_time = time.time()

    # Khởi tạo DuckDB Engine
    conn = duckdb.connect(database=':memory:')

    # -------------------------------------------------------------------------
    # BƯỚC 1: Làm sạch bảng ADMISSIONS (Thông tin nhập viện)
    # Logic: Ép kiểu ID, đảm bảo thời gian hợp lệ (ngày ra viện phải sau ngày nhập viện)
    # -------------------------------------------------------------------------
    silver_admissions_dir = os.path.join(silver_dir, "hosp", "admissions")
    os.makedirs(silver_admissions_dir, exist_ok=True)

    # Đường dẫn file Bronze đầu vào
    bronze_admissions_path = os.path.join(bronze_dir, "hosp", "admissions", "admissions.parquet")

    query_admissions = f"""
        COPY (
            SELECT
                CAST(subject_id AS INTEGER) AS subject_id,
                CAST(hadm_id AS INTEGER) AS hadm_id,
                CAST(admittime AS TIMESTAMP) AS admittime,
                CAST(dischtime AS TIMESTAMP) AS dischtime,
                CAST(deathtime AS TIMESTAMP) AS deathtime,
                admission_type,
                admission_location,
                discharge_location,
                insurance,
                language,
                marital_status,
                race,
                hospital_expire_flag
            FROM '{bronze_admissions_path}'
            WHERE admittime <= dischtime -- Loại bỏ lỗi logic về thời gian
              AND subject_id IS NOT NULL
              AND hadm_id IS NOT NULL
        ) TO '{silver_admissions_dir}/admissions.parquet' (FORMAT PARQUET);
    """
    print("Đang làm sạch bảng: admissions...")
    conn.execute(query_admissions)

    # -------------------------------------------------------------------------
    # BƯỚC 2: Làm sạch bảng CHARTEVENTS (Dấu hiệu sinh tồn - Đã được Partition từ Bronze)
    # Logic: Đọc từ cấu trúc partition Bronze, ép kiểu, lọc bỏ các giá trị sinh lý bất thường (Outliers)
    # -------------------------------------------------------------------------
    silver_chartevents_dir = os.path.join(silver_dir, "icu", "chartevents")
    os.makedirs(silver_chartevents_dir, exist_ok=True)

    # Trỏ vào toàn bộ các file parquet trong các thư mục partition ở Bronze
    bronze_chartevents_pattern = os.path.join(bronze_dir, "icu", "chartevents", "*", "*.parquet")

    query_chartevents = f"""
        COPY (
            SELECT
                CAST(subject_id AS INTEGER) AS subject_id,
                CAST(hadm_id AS INTEGER) AS hadm_id,
                CAST(stay_id AS INTEGER) AS stay_id,
                CAST(charttime AS TIMESTAMP) AS charttime,
                CAST(itemid AS INTEGER) AS itemid,
                CAST(valuenum AS FLOAT) AS valuenum, -- Ép kiểu chỉ số đo về dạng số thực
                valueuom AS unit_of_measure
            FROM '{bronze_chartevents_pattern}'
            WHERE valuenum IS NOT NULL
              AND valuenum > 0 -- Loại bỏ các giá trị đo bằng 0 hoặc âm vô lý
        ) TO '{silver_chartevents_dir}' (FORMAT PARQUET, PARTITION_BY itemid, OVERWRITE_OR_IGNORE 1);
    """
    print("Đang làm sạch bảng lớn: chartevents (giữ nguyên cấu trúc Partition)...")
    conn.execute(query_chartevents)

    duration = time.time() - start_time
    print(f"Tầng Silver đã hoàn thành trong {duration:.2f} giây!")
    print(f"Dữ liệu sạch được lưu tại: {silver_dir}")

# Kích hoạt pipeline tầng Silver
build_silver_layer(BRONZE_BASE_DIR, SILVER_BASE_DIR)

[Silver Layer] Bắt đầu quá trình làm sạch và chuẩn hóa dữ liệu...
Đang làm sạch bảng: admissions...
Đang làm sạch bảng lớn: chartevents (giữ nguyên cấu trúc Partition)...
Tầng Silver đã hoàn thành trong 1.85 giây!
Dữ liệu sạch được lưu tại: ./data/silver


# **Tầng 3: Gold - Thực hiện việc chuẩn bị dữ liệu cho các bài toán đầu ra**

Hiện thực hóa 2 bảng Gold quan trọng nhất bằng DuckDB SQL:

- **ICU Cohort Matrix Table:** Bảng phẳng (Flat table) chứa thông tin tổng quan của mỗi ca nằm viện ICU.

- **Vitals Time-Series Aggregations (Block 4 giờ)**: Gom tụ dữ liệu nhịp tim/huyết áp dày đặc thành các block thời gian để phục vụ train model.

## **Kỹ thuật Cửa số Window Function & Date Truncation**

- **Với bảng Cohort Matrix:** Bạn phải liên kết (`JOIN`) 3 tầng dữ liệu: Bệnh nhân (`patients`) $\rightarrow$ Lần nhập viện (`admissions`) $\rightarrow$ Lần vào khoa hồi sức (`icustays`). Thách thức lớn nhất ở đây là tính toán chính xác số tuổi của bệnh nhân lúc nhập viện dựa trên năm sinh và năm nhập viện.
- **Với bảng Time-Series (Block 4 giờ):** `chartevents` có hàng triệu dòng đo đạc theo từng phút. Để gom chúng thành các block 4 giờ, ta sử dụng kỹ thuật làm tròn thời gian (date_trunc hoặc phép toán chia giờ) kết hợp với hàm gộp `MIN/MAX/AVG`.

In [7]:
# Định nghĩa các đường dẫn tầng dữ liệu
SILVER_BASE_DIR = "./data/silver"
GOLD_BASE_DIR = "./data/gold"

def build_gold_layer(silver_dir, gold_dir):
    print("[Gold Layer] Bắt đầu tổng hợp dữ liệu phân tích nâng cao...")
    start_time = time.time()

    # Kết nối DuckDB Engine
    conn = duckdb.connect(database=':memory:')

    # Định nghĩa đường dẫn các file Silver đầu vào
    admissions_parquet = os.path.join(silver_dir, "hosp", "admissions", "admissions.parquet")
    chartevents_pattern = os.path.join(silver_dir, "icu", "chartevents", "*", "*.parquet")

    # Giả định Tiến đã có file silver cho patients và icustays (nếu chưa có ta sẽ đọc tạm từ bronze)
    # Để đảm bảo code chạy mượt, ta khai báo đường dẫn trực tiếp:
    patients_parquet = os.path.join(silver_dir, "hosp", "patients", "patients.parquet")
    icustays_parquet = os.path.join(silver_dir, "icu", "icustays", "icustays.parquet")

    # Mẹo nhỏ: Nếu Tiến chưa chạy bài làm sạch cho patients/icustays ở tuần 2,
    # ta có thể cấu hình đọc từ tầng bronze để test logic tầng Gold trước.
    if not os.path.exists(patients_parquet): patients_parquet = os.path.join("./data/bronze", "hosp", "patients", "patients.parquet")
    if not os.path.exists(icustays_parquet): icustays_parquet = os.path.join("./data/bronze", "icu", "icustays", "icustays.parquet")

    # -------------------------------------------------------------------------
    # BƯỚC 1: Tạo bảng ICU_COHORT_MATRIX
    # Tính toán: Tuổi khi nhập viện, Thời gian nằm viện (Length of Stay - LOS tính bằng ngày)
    # -------------------------------------------------------------------------
    gold_cohort_dir = os.path.join(gold_dir, "clinical_marts", "icu_cohort_matrix")
    os.makedirs(gold_cohort_dir, exist_ok=True)

    query_cohort = f"""
        COPY (
            SELECT
                ie.subject_id,
                ie.hadm_id,
                ie.stay_id,
                p.gender,
                -- Tính tuổi: Lấy năm của ngày nhập viện trừ đi năm sinh
                EXTRACT(YEAR FROM adm.admittime) - p.anchor_age AS birth_year_estimated,
                p.anchor_age + (EXTRACT(YEAR FROM adm.admittime) - p.anchor_year) AS age_at_admission,
                adm.admission_type,
                adm.race,
                ie.intime AS icu_intime,
                ie.outtime AS icu_outtime,
                -- Tính số ngày nằm ICU (Length of Stay)
                ROUND(CAST(EXTRACT(EPOCH FROM (ie.outtime - ie.intime)) / 86400 AS NUMERIC), 2) AS icu_los_days,
                adm.hospital_expire_flag AS died_in_hospital
            FROM '{icustays_parquet}' ie
            JOIN '{admissions_parquet}' adm ON ie.hadm_id = adm.hadm_id
            JOIN '{patients_parquet}' p ON ie.subject_id = p.subject_id
            WHERE (p.anchor_age + (EXTRACT(YEAR FROM adm.admittime) - p.anchor_year)) >= 18 -- Chỉ lấy bệnh nhân người lớn
        ) TO '{gold_cohort_dir}/icu_cohort_matrix.parquet' (FORMAT PARQUET);
    """
    print("Đang xây dựng Mart: icu_cohort_matrix...")
    conn.execute(query_cohort)

    # -------------------------------------------------------------------------
    # BƯỚC 2: Tạo bảng VITALS_4HR_AGGREGATIONS (Dữ liệu tim mạch/sinh tồn block 4 giờ)
    # Logic: Làm tròn thời gian về mốc 4 giờ (0h, 4h, 8h...) và tính toán thống kê
    # -------------------------------------------------------------------------
    gold_vitals_dir = os.path.join(gold_dir, "clinical_marts", "vitals_4hr_aggregations")
    os.makedirs(gold_vitals_dir, exist_ok=True)

    query_vitals = f"""
        COPY (
            SELECT
                stay_id,
                itemid,
                -- Logic làm tròn thời gian về block 4 giờ bằng DuckDB SQL
                -- Lấy thời gian gốc trừ đi số phút dư thừa để đưa về mốc 4 giờ gần nhất
                DATE_TRUNC('hour', charttime) - CAST(EXTRACT(HOUR FROM charttime) % 4 || ' hours' AS INTERVAL) AS block_4hr_start,
                MIN(valuenum) AS vital_min,
                MAX(valuenum) AS vital_max,
                ROUND(CAST(AVG(valuenum) AS NUMERIC), 2) AS vital_avg,
                COUNT(valuenum) AS measurement_count
            FROM '{chartevents_pattern}'
            WHERE itemid IN (220045, 220179) -- 220045: Heart Rate, 220179: Non Invasive Blood Pressure Systolic
            GROUP BY stay_id, itemid, block_4hr_start
        ) TO '{gold_vitals_dir}/vitals_4hr_aggregations.parquet' (FORMAT PARQUET);
    """
    print("Đang xây dựng Mart: vitals_4hr_aggregations (Tải trọng lớn)...")
    conn.execute(query_vitals)

    duration = time.time() - start_time
    print(f"Toàn bộ kiến trúc Medallion (Tầng Gold) đã hoàn thành trong {duration:.2f} giây!")
    print(f"Thành phẩm sẵn sàng cho AI/Data Science tại: {gold_dir}")


# Chạy pipeline tầng Gold
build_gold_layer(SILVER_BASE_DIR, GOLD_BASE_DIR)

[Gold Layer] Bắt đầu tổng hợp dữ liệu phân tích nâng cao...
Đang xây dựng Mart: icu_cohort_matrix...
Đang xây dựng Mart: vitals_4hr_aggregations (Tải trọng lớn)...
Toàn bộ kiến trúc Medallion (Tầng Gold) đã hoàn thành trong 0.07 giây!
Thành phẩm sẵn sàng cho AI/Data Science tại: ./data/gold


In [8]:
# Xem thử 5 dòng dữ liệu chuỗi thời gian đã được tổng hợp theo block 4 giờ
conn = duckdb.connect(database=':memory:')
preview_gold = conn.execute(f"SELECT * FROM './data/gold/clinical_marts/vitals_4hr_aggregations/vitals_4hr_aggregations.parquet' LIMIT 5").df()
preview_gold

,stay_id,itemid,block_4hr_start,vital_min,vital_max,vital_avg,measurement_count
0,32604416,220179,2132-12-17 00:00:00,93.0,119.0,109.00,3
1,32604416,220179,2132-12-17 08:00:00,90.0,109.0,99.75,4
2,32604416,220179,2132-12-17 12:00:00,107.0,118.0,111.75,4
3,32604416,220179,2132-12-16 08:00:00,118.0,118.0,118.00,1
4,36084484,220179,2142-08-31 16:00:00,111.0,125.0,117.33,3
